# Baseline: Aggregates + CatBoost

Prefix-safe Kaggle baseline for low-label comparison against `kaggle_forecast_pipeline.ipynb`. It uses the same entity split protocol, deterministic forecast-style prefix cuts, train-only aggregate statistics, and standardized low-label result artifacts.


In [ ]:
# Cell 1 - Setup
import hashlib
import json
import pathlib
import re
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "/kaggle/working/denoising-event-sequences", "--quiet"],
    check=True,
)
sys.path.insert(0, "/kaggle/working/denoising-event-sequences")

try:
    from catboost import CatBoostClassifier, Pool
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "catboost", "--quiet"], check=True)
    from catboost import CatBoostClassifier, Pool

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)

from src.data.splits import make_entity_splits

SMOKE_RUN = False
print("Setup complete")


In [ ]:
# Cell 2 - Paths and config
WORKING_DIR = pathlib.Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
OUTPUT_DIR = WORKING_DIR / "outputs" / "baselines" / "aggregates_catboost"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_AMP = False
config = {
    "data": {
        "max_seq_len": 256,
        "min_seq_len": 2,
        "event_type_col": "MCC",
        "timestamp_col": "TRDATETIME",
        "target_col": "target_flag",
        "numerical_cols": ["amount"],
        "categorical_cols": ["channel_type", "trx_category"],
        "group_col": "cl_id",
        "truncation_pretrain": "sliding_window",
        "truncation_eval": "last_events",
        "amount_transform": "robust_scaler",
        "time_transform": "none",
        "train_ratio": 0.70,
        "val_ratio": 0.15,
        "test_ratio": 0.15,
        "min_vocab_count": 5,
    },
    "model": {
        "event_type_emb_dim": 64,
        "cat_emb_dim": 32,
        "num_projection_dim": 64,
        "time_projection_dim": 64,
        "hidden_dim": 256,
        "num_layers": 4,
        "num_heads": 8,
        "dim_feedforward": 1024,
        "dropout": 0.10,
        "activation": "gelu",
        "max_seq_len": 256,
    },
    "pooling": {"type": "gated_attention"},
    "forecasting": {
        "cut_min_ratio": 0.50,
        "cut_max_ratio": 0.80,
        "min_future_events": 2,
        "cut_seed": 42,
    },
    "training": {
        "batch_size": 128,
        "num_epochs_finetune": 20,
        "lr": 3e-4,
        "lr_encoder": 3e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "gradient_clip_val": 1.0,
        "mixed_precision": USE_AMP,
        "early_stopping_patience": 5,
        "log_every_n_steps": 50,
    },
}

LABEL_FRACTIONS = [1.00] if SMOKE_RUN else [0.05, 0.25, 0.50, 0.75, 1.00]
LABEL_SAMPLING_SEEDS = [42] if SMOKE_RUN else [42, 43, 44, 45, 46]
METRIC_COLS = ["accuracy", "macro_f1", "weighted_f1", "roc_auc", "pr_auc", "balanced_accuracy"]

print("Output dir:", OUTPUT_DIR)
print("Cutoff policy:", config["forecasting"])


In [ ]:
# Cell 3 - Load data, shared splits, and prefix-only events
GROUP_COL = config["data"]["group_col"]
TARGET_COL = config["data"]["target_col"]
TIME_COL = config["data"]["timestamp_col"]
EVENT_COL = config["data"]["event_type_col"]
NUM_COL = "amount"
CAT_COLS = config["data"]["categorical_cols"]

def _forecast_cut_bounds(n_events, cfg):
    f_cfg = cfg.get("forecasting", {})
    min_ratio = float(f_cfg.get("cut_min_ratio", 0.50))
    max_ratio = float(f_cfg.get("cut_max_ratio", 0.80))
    min_future = int(f_cfg.get("min_future_events", 2))
    if n_events < min_future + 1:
        return None
    cut_low = max(1, int(np.floor(n_events * min_ratio)))
    cut_high = min(n_events - min_future, int(np.floor(n_events * max_ratio)))
    if cut_high < cut_low:
        cut_low = max(1, n_events - min_future)
        cut_high = n_events - min_future
    if cut_high < cut_low:
        return None
    return cut_low, cut_high


def _stable_entity_seed(entity_id, base_seed=42):
    digest = hashlib.blake2b(f"{base_seed}:{entity_id}".encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, byteorder="little", signed=False)


def deterministic_forecast_cut(entity_id, n_events, cfg, base_seed=None):
    bounds = _forecast_cut_bounds(int(n_events), cfg)
    if bounds is None:
        return None
    cut_low, cut_high = bounds
    if base_seed is None:
        base_seed = int(cfg.get("forecasting", {}).get("cut_seed", 42))
    rng = np.random.default_rng(_stable_entity_seed(entity_id, base_seed))
    return int(rng.integers(cut_low, cut_high + 1))


def build_prefix_events(df_source, entity_ids, cfg):
    entity_set = set(entity_ids)
    working = df_source[df_source[GROUP_COL].isin(entity_set)].copy()
    working = working.sort_values([GROUP_COL, TIME_COL], kind="stable").reset_index(drop=True)
    working["forecast_event_pos"] = working.groupby(GROUP_COL, sort=False).cumcount()
    counts = working.groupby(GROUP_COL, sort=False).size()
    cut_by_entity = {}
    for entity_id in entity_ids:
        if entity_id not in counts.index:
            continue
        cut = deterministic_forecast_cut(entity_id, int(counts.loc[entity_id]), cfg)
        if cut is not None:
            cut_by_entity[entity_id] = cut
    working["forecast_cut"] = working[GROUP_COL].map(cut_by_entity)
    prefix = working[
        working["forecast_cut"].notna()
        & (working["forecast_event_pos"] < working["forecast_cut"])
    ].copy()
    prefix["forecast_cut"] = prefix["forecast_cut"].astype(int)
    valid_entity_ids = [entity_id for entity_id in entity_ids if entity_id in cut_by_entity]
    return prefix.reset_index(drop=True), cut_by_entity, valid_entity_ids


def filter_splits_to_valid(splits_in, valid_entity_ids):
    valid = set(valid_entity_ids)
    filtered = {name: [entity_id for entity_id in ids if entity_id in valid] for name, ids in splits_in.items()}
    if any(len(ids) == 0 for ids in filtered.values()):
        raise ValueError(f"Empty split after prefix filtering: { {k: len(v) for k, v in filtered.items()} }")
    return filtered


def assert_no_future_leakage(prefix_df, cut_by_entity):
    counts = prefix_df.groupby(GROUP_COL, sort=False).size()
    for entity_id, cut in cut_by_entity.items():
        if entity_id not in counts.index:
            continue
        observed = int(counts.loc[entity_id])
        if observed != int(cut):
            raise AssertionError(f"Prefix length mismatch for {entity_id}: {observed} != cut {cut}")
    max_pos = prefix_df.groupby(GROUP_COL, sort=False)["forecast_event_pos"].max()
    bad = [entity_id for entity_id, pos in max_pos.items() if int(pos) >= int(cut_by_entity[entity_id])]
    if bad:
        raise AssertionError(f"Future events leaked into prefix for entities: {bad[:5]}")


df = pd.read_csv(DATA_DIR / "rosbank" / "train.csv.gz")
df[TIME_COL] = pd.to_datetime(df[TIME_COL], format="%d%b%y:%H:%M:%S")
labels_by_entity = df.groupby(GROUP_COL)[TARGET_COL].first().to_dict()

raw_splits = make_entity_splits(
    df,
    entity_col=GROUP_COL,
    target_col=TARGET_COL,
    train_ratio=config["data"]["train_ratio"],
    val_ratio=config["data"]["val_ratio"],
    test_ratio=config["data"]["test_ratio"],
    seed=42,
)
raw_entity_ids = raw_splits["train"] + raw_splits["val"] + raw_splits["test"]
df_prefix, forecast_cut_by_entity, valid_entity_ids = build_prefix_events(df, raw_entity_ids, config)
splits = filter_splits_to_valid(raw_splits, valid_entity_ids)
all_entity_ids = splits["train"] + splits["val"] + splits["test"]
df_prefix = df_prefix[df_prefix[GROUP_COL].isin(set(all_entity_ids))].copy()
forecast_cut_by_entity = {entity_id: forecast_cut_by_entity[entity_id] for entity_id in all_entity_ids}
assert_no_future_leakage(df_prefix, forecast_cut_by_entity)

print("Raw events:", df.shape)
print("Prefix events:", df_prefix.shape)
print("Split sizes:", {k: len(v) for k, v in splits.items()})
print("Filtered short entities:", len(raw_entity_ids) - len(all_entity_ids))


In [ ]:
# Cell 4 - Shared sampling, metrics, aggregation, and plotting helpers
def get_entity_labels(df_source, group_col, target_col):
    return df_source.groupby(group_col)[target_col].first().to_dict()


def sample_labeled_entities(train_ids, labels_by_entity, fraction, seed):
    train_ids = list(train_ids)
    if fraction >= 1.0:
        return train_ids

    rng = np.random.default_rng(seed)
    by_label = {}
    for entity_id in train_ids:
        label = int(labels_by_entity[entity_id])
        by_label.setdefault(label, []).append(entity_id)

    sampled = []
    for label, ids in sorted(by_label.items()):
        ids = np.array(ids, dtype=object)
        n = max(1, int(round(len(ids) * fraction)))
        n = min(n, len(ids))
        sampled.extend(rng.choice(ids, size=n, replace=False).tolist())

    rng.shuffle(sampled)
    return sampled


def compute_binary_metrics(y_true, pos_proba):
    y_true = np.asarray(y_true).astype(int)
    pos_proba = np.asarray(pos_proba).astype(float)
    y_pred = (pos_proba >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, pos_proba)),
        "pr_auc": float(average_precision_score(y_true, pos_proba)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def json_default(obj):
    if isinstance(obj, pathlib.Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        value = float(obj)
        return None if not np.isfinite(value) else value
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if pd.isna(obj):
        return None
    return str(obj)


def aggregate_low_label_runs(run_rows, metric_cols=METRIC_COLS):
    runs_df = pd.DataFrame(run_rows)
    if runs_df.empty:
        return runs_df, []
    agg_rows = []
    for (pipeline, fraction), group in runs_df.groupby(["pipeline", "fraction"], sort=True):
        row = {"pipeline": pipeline, "fraction": float(fraction), "n_seeds": int(group["seed"].nunique())}
        for metric in metric_cols:
            values = pd.to_numeric(group[metric], errors="coerce").dropna()
            if len(values):
                row[f"{metric}_mean"] = float(values.mean())
                row[f"{metric}_std"] = float(values.std(ddof=0))
        agg_rows.append(row)
    agg_df = pd.DataFrame(agg_rows).sort_values(["fraction", "pipeline"]).reset_index(drop=True)
    return agg_df, agg_rows


def build_summary_table(agg_rows):
    rows = []
    for row in agg_rows:
        rows.append(
            {
                "pipeline": row["pipeline"],
                "fraction": row["fraction"],
                "n_seeds": row["n_seeds"],
                "roc_auc": f"{row.get('roc_auc_mean', float('nan')):.4f} +/- {row.get('roc_auc_std', 0.0):.4f}",
                "pr_auc": f"{row.get('pr_auc_mean', float('nan')):.4f} +/- {row.get('pr_auc_std', 0.0):.4f}",
                "macro_f1": f"{row.get('macro_f1_mean', float('nan')):.4f} +/- {row.get('macro_f1_std', 0.0):.4f}",
                "balanced_accuracy": f"{row.get('balanced_accuracy_mean', float('nan')):.4f} +/- {row.get('balanced_accuracy_std', 0.0):.4f}",
            }
        )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["fraction", "pipeline"]).reset_index(drop=True)


def plot_low_label_metrics(agg_df, output_dir, metrics=("roc_auc", "pr_auc", "macro_f1")):
    plot_dir = output_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    plot_paths = {}
    if agg_df.empty:
        return plot_paths
    for metric in metrics:
        mean_col = f"{metric}_mean"
        std_col = f"{metric}_std"
        if mean_col not in agg_df.columns:
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        for pipeline, group in agg_df.groupby("pipeline", sort=True):
            group = group.sort_values("fraction")
            x = pd.to_numeric(group["fraction"], errors="coerce")
            y = pd.to_numeric(group[mean_col], errors="coerce")
            yerr = pd.to_numeric(group.get(std_col, 0.0), errors="coerce").fillna(0.0)
            ax.errorbar(x, y, yerr=yerr, marker="o", linewidth=2, capsize=4, label=pipeline)
        ax.set_title(f"{metric} by label fraction")
        ax.set_xlabel("Label fraction")
        ax.set_ylabel(metric)
        ax.set_ylim(0.0, 1.0)
        ax.grid(True, alpha=0.25)
        ax.legend()
        path = plot_dir / f"{metric}_by_fraction.png"
        fig.savefig(path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        plot_paths[metric] = path
    return plot_paths


def save_low_label_outputs(run_rows, prediction_rows, baseline_name, output_dir, extra_artifacts=None):
    output_dir.mkdir(parents=True, exist_ok=True)
    all_runs_df = pd.DataFrame(run_rows)
    if not all_runs_df.empty:
        all_runs_df = all_runs_df.sort_values(["pipeline", "fraction", "seed"]).reset_index(drop=True)
    all_runs_path = output_dir / "low_label_all_runs.csv"
    all_runs_df.to_csv(all_runs_path, index=False)

    agg_df, agg_rows = aggregate_low_label_runs(run_rows)
    agg_path = output_dir / "low_label_aggregate.csv"
    agg_json_path = output_dir / "low_label_aggregate.json"
    agg_df.to_csv(agg_path, index=False)
    agg_json_path.write_text(json.dumps(agg_rows, indent=2, ensure_ascii=False, default=json_default))

    summary_df = build_summary_table(agg_rows)
    summary_path = output_dir / "low_label_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    predictions_df = pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame()
    pred_path = output_dir / "test_predictions.csv"
    predictions_df.to_csv(pred_path, index=False)

    plot_paths = plot_low_label_metrics(agg_df, output_dir)
    metrics_payload = {
        "baseline": baseline_name,
        "cutoff_policy": config.get("forecasting", {}),
        "metric_columns": METRIC_COLS,
        "split_sizes": {k: len(v) for k, v in splits.items()},
        "artifacts": {
            "low_label_all_runs": all_runs_path,
            "low_label_aggregate": agg_path,
            "low_label_aggregate_json": agg_json_path,
            "low_label_summary": summary_path,
            "test_predictions": pred_path,
            "plots": plot_paths,
            **(extra_artifacts or {}),
        },
        "aggregate": agg_rows,
    }
    metrics_path = output_dir / "metrics.json"
    metrics_path.write_text(json.dumps(metrics_payload, indent=2, ensure_ascii=False, default=json_default))

    print("Low-label results saved:")
    for path in [all_runs_path, agg_path, agg_json_path, summary_path, pred_path, metrics_path]:
        print(" -", path)
    for path in plot_paths.values():
        print(" -", path)
    print("\nLow-label summary:")
    print(summary_df.to_string(index=False))
    try:
        display(summary_df)
    except NameError:
        pass
    return all_runs_df, agg_df, summary_df, predictions_df, metrics_payload


In [ ]:
# Cell 5 - Prefix-safe aggregate feature builder
CATBOOST_CATEGORICAL_FEATURES = []
CATBOOST_FITTED_FEATURE_METADATA = {}


def safe_name(value):
    return re.sub(r"[^0-9A-Za-z_]+", "_", str(value)).strip("_")[:60]


def safe_divide(num, denom):
    return num.div(denom.replace(0, np.nan), axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)


def entropy_from_counts(counts):
    totals = counts.sum(axis=1).replace(0, np.nan)
    probs = counts.div(totals, axis=0).fillna(0.0)
    log_probs = np.log(probs.replace(0.0, np.nan)).fillna(0.0)
    return -(probs * log_probs).sum(axis=1)


def add_entropy_features(features, df_source, col, prefix):
    if col not in df_source.columns:
        return features
    counts = pd.crosstab(df_source[GROUP_COL], df_source[col].astype(str)).reindex(features.index, fill_value=0)
    totals = counts.sum(axis=1).replace(0, np.nan)
    sorted_counts = np.sort(counts.to_numpy(), axis=1)[:, ::-1]
    features[f"{prefix}_entropy"] = entropy_from_counts(counts)
    features[f"{prefix}_top1_share"] = pd.Series(sorted_counts[:, :1].sum(axis=1), index=features.index).div(totals).fillna(0.0)
    features[f"{prefix}_top3_share"] = pd.Series(sorted_counts[:, :3].sum(axis=1), index=features.index).div(totals).fillna(0.0)
    return features


def fit_top_values(df_train_prefix, col, top_n):
    if col not in df_train_prefix.columns:
        return []
    return df_train_prefix[col].astype(str).value_counts().head(top_n).index.tolist()


def add_top_count_features(features, df_source, col, top_values, prefix=None, add_counts=True, denom_col="event_count"):
    if not top_values or col not in df_source.columns:
        return features
    prefix = prefix or col
    tmp = df_source[[GROUP_COL, col]].copy()
    tmp[col] = tmp[col].astype(str)
    tmp = tmp[tmp[col].isin(top_values)]
    counts = pd.crosstab(tmp[GROUP_COL], tmp[col])
    counts = counts.reindex(index=features.index, columns=top_values, fill_value=0)
    counts.columns = [f"{prefix}_top_{safe_name(v)}_cnt" for v in top_values]
    denom = features[denom_col] if denom_col in features.columns else features["event_count"]
    freqs = counts.div(denom.replace(0, np.nan), axis=0).fillna(0.0)
    freqs.columns = [c.replace("_cnt", "_freq") for c in counts.columns]
    return features.join(counts if add_counts else pd.DataFrame(index=features.index)).join(freqs)


def add_recency_weighted_top_features(features, df_source, col, top_values, half_life_days):
    if not top_values or col not in df_source.columns:
        return features
    weight_col = f"recency_weight_{half_life_days}d"
    tmp = df_source[[GROUP_COL, col, weight_col]].copy()
    tmp[col] = tmp[col].astype(str)
    tmp = tmp[tmp[col].isin(top_values)]
    weighted = tmp.pivot_table(index=GROUP_COL, columns=col, values=weight_col, aggfunc="sum", fill_value=0.0)
    weighted = weighted.reindex(index=features.index, columns=top_values, fill_value=0.0)
    weighted.columns = [f"recency_weighted_{half_life_days}d_{col}_top_{safe_name(v)}_sum" for v in top_values]
    denom = features[f"recency_weight_{half_life_days}d_sum"].replace(0, np.nan)
    freqs = weighted.div(denom, axis=0).fillna(0.0)
    freqs.columns = [c.replace("_sum", "_freq") for c in weighted.columns]
    return features.join(weighted).join(freqs)


def add_subset_aggregates(features, df_source, prefix):
    if df_source.empty:
        return features
    g = df_source.groupby(GROUP_COL, sort=False)
    sub = g.agg(
        **{
            f"{prefix}_event_count": (EVENT_COL, "size"),
            f"{prefix}_amount_sum": (NUM_COL, "sum"),
            f"{prefix}_amount_mean": (NUM_COL, "mean"),
            f"{prefix}_amount_std": (NUM_COL, "std"),
            f"{prefix}_amount_abs_sum": ("amount_abs", "sum"),
            f"{prefix}_amount_abs_mean": ("amount_abs", "mean"),
            f"{prefix}_positive_share_num": ("is_pos", "mean"),
            f"{prefix}_negative_share_num": ("is_neg", "mean"),
            f"{prefix}_unique_event_types": (EVENT_COL, "nunique"),
            f"{prefix}_active_days": ("date", "nunique"),
        }
    )
    return features.join(sub.reindex(features.index))


def mode_as_string(values):
    values = values.astype(str)
    if values.empty:
        return "<NA>"
    mode = values.mode(dropna=False)
    return str(mode.iloc[0]) if len(mode) else "<NA>"


def assert_catboost_feature_frame_is_leak_safe(features, df_prefix_source, split_ids, cut_by_entity):
    train_set, val_set, test_set = map(set, (split_ids["train"], split_ids["val"], split_ids["test"]))
    assert train_set.isdisjoint(val_set)
    assert train_set.isdisjoint(test_set)
    assert val_set.isdisjoint(test_set)
    assert_no_future_leakage(df_prefix_source, cut_by_entity)
    forbidden = {TARGET_COL, TIME_COL, "forecast_cut", "forecast_event_pos", "target"}
    leaked = sorted(forbidden.intersection(features.columns))
    if leaked:
        raise AssertionError(f"Forbidden leakage-prone feature columns: {leaked}")
    expected_ids = train_set | val_set | test_set
    if set(features.index) != expected_ids:
        raise AssertionError("Feature index does not match split entities")


def build_aggregate_features(df_prefix_source, all_ids, train_ids):
    working = df_prefix_source.copy()
    working = working.sort_values([GROUP_COL, TIME_COL], kind="stable")
    working[NUM_COL] = pd.to_numeric(working[NUM_COL], errors="coerce").fillna(0.0)
    working["amount_abs"] = working[NUM_COL].abs()
    working["amount_pos"] = working[NUM_COL].clip(lower=0.0)
    working["amount_neg"] = working[NUM_COL].clip(upper=0.0)
    working["is_pos"] = (working[NUM_COL] > 0).astype(int)
    working["is_neg"] = (working[NUM_COL] < 0).astype(int)
    working["date"] = working[TIME_COL].dt.date
    working["hour"] = working[TIME_COL].dt.hour
    working["dayofweek"] = working[TIME_COL].dt.dayofweek
    working["month"] = working[TIME_COL].dt.month
    working["is_weekend"] = working["dayofweek"].isin([5, 6]).astype(int)
    working["gap_days"] = working.groupby(GROUP_COL, sort=False)[TIME_COL].diff().dt.total_seconds().div(86400).fillna(0.0).clip(lower=0.0)
    working["entity_last_prefix_time"] = working.groupby(GROUP_COL, sort=False)[TIME_COL].transform("max")
    working["days_before_prefix_end"] = working["entity_last_prefix_time"].sub(working[TIME_COL]).dt.total_seconds().div(86400).fillna(0.0).clip(lower=0.0)
    for half_life in [14, 30, 90]:
        working[f"recency_weight_{half_life}d"] = np.exp(-np.log(2.0) * working["days_before_prefix_end"] / float(half_life))
        working[f"amount_abs_weighted_{half_life}d"] = working["amount_abs"] * working[f"recency_weight_{half_life}d"]
    working["prefix_event_pos"] = working.groupby(GROUP_COL, sort=False).cumcount()
    working["prefix_event_count"] = working.groupby(GROUP_COL, sort=False)[EVENT_COL].transform("size")
    working["is_last_half"] = working["prefix_event_pos"] >= (working["prefix_event_count"] / 2.0)
    working["prev_event"] = working.groupby(GROUP_COL, sort=False)[EVENT_COL].shift(1).astype(str)
    working["transition"] = working["prev_event"] + "__" + working[EVENT_COL].astype(str)
    working.loc[working["prev_event"].eq("nan"), "transition"] = np.nan

    train_set = set(train_ids)
    train_df = working[working[GROUP_COL].isin(train_set)]
    reference_time = train_df[TIME_COL].max()

    g = working.groupby(GROUP_COL, sort=False)
    features = g.agg(
        event_count=(EVENT_COL, "size"),
        amount_sum=(NUM_COL, "sum"),
        amount_mean=(NUM_COL, "mean"),
        amount_std=(NUM_COL, "std"),
        amount_min=(NUM_COL, "min"),
        amount_max=(NUM_COL, "max"),
        amount_median=(NUM_COL, "median"),
        amount_abs_sum=("amount_abs", "sum"),
        amount_abs_mean=("amount_abs", "mean"),
        amount_abs_max=("amount_abs", "max"),
        amount_pos_sum=("amount_pos", "sum"),
        amount_neg_sum=("amount_neg", "sum"),
        positive_event_count=("is_pos", "sum"),
        negative_event_count=("is_neg", "sum"),
        first_prefix_time=(TIME_COL, "min"),
        last_prefix_time=(TIME_COL, "max"),
        active_days=("date", "nunique"),
        unique_event_types=(EVENT_COL, "nunique"),
        gap_days_mean=("gap_days", "mean"),
        gap_days_std=("gap_days", "std"),
        gap_days_median=("gap_days", "median"),
        gap_days_max=("gap_days", "max"),
        recent_gap_days=("gap_days", "last"),
        hour_mean=("hour", "mean"),
        hour_std=("hour", "std"),
        dayofweek_mean=("dayofweek", "mean"),
        weekend_share=("is_weekend", "mean"),
    )

    for q in [0.10, 0.25, 0.75, 0.90]:
        suffix = int(q * 100)
        features[f"amount_q{suffix}"] = g[NUM_COL].quantile(q)
        features[f"amount_abs_q{suffix}"] = g["amount_abs"].quantile(q)
        features[f"gap_days_q{suffix}"] = g["gap_days"].quantile(q)

    for half_life in [14, 30, 90]:
        features[f"recency_weight_{half_life}d_sum"] = g[f"recency_weight_{half_life}d"].sum()
        features[f"recency_amount_abs_{half_life}d_sum"] = g[f"amount_abs_weighted_{half_life}d"].sum()
        features[f"recency_amount_abs_{half_life}d_mean"] = safe_divide(
            features[f"recency_amount_abs_{half_life}d_sum"],
            features[f"recency_weight_{half_life}d_sum"],
        )

    raw_cat_cols = [col for col in [EVENT_COL, *CAT_COLS, "currency"] if col in working.columns]
    cat_feature_names = []
    for col in raw_cat_cols:
        if col in working.columns:
            features[f"unique_{col}"] = g[col].nunique()
            features[f"first_{col}"] = g[col].first().astype(str)
            features[f"last_{col}"] = g[col].last().astype(str)
            features[f"mode_{col}"] = g[col].agg(mode_as_string).astype(str)
            cat_feature_names.extend([f"first_{col}", f"last_{col}", f"mode_{col}"])
    features["last_transition"] = g["transition"].last().fillna("<NA>").astype(str)
    cat_feature_names.append("last_transition")

    features["prefix_span_days"] = (
        features["last_prefix_time"] - features["first_prefix_time"]
    ).dt.total_seconds().div(86400).fillna(0.0)
    features["events_per_active_day"] = features["event_count"] / features["active_days"].replace(0, np.nan)
    features["events_per_span_day"] = features["event_count"] / (features["prefix_span_days"] + 1.0)
    features["positive_event_share"] = features["positive_event_count"] / features["event_count"].replace(0, np.nan)
    features["negative_event_share"] = features["negative_event_count"] / features["event_count"].replace(0, np.nan)
    features["amount_abs_per_event"] = features["amount_abs_sum"] / features["event_count"].replace(0, np.nan)
    features["days_since_last_prefix_event_train_ref"] = (reference_time - features["last_prefix_time"]).dt.total_seconds().div(86400).clip(lower=0.0).fillna(0.0)
    features = features.drop(columns=["first_prefix_time", "last_prefix_time"])

    top_specs = [(EVENT_COL, 80), ("channel_type", 20), ("trx_category", 40)]
    if "currency" in working.columns:
        top_specs.append(("currency", 20))
    fitted_top_values = []
    for col, top_n in top_specs:
        top_values = fit_top_values(train_df, col, top_n)
        fitted_top_values.append((col, top_values))
        features = add_top_count_features(features, working, col, top_values)
        for half_life in [14, 30, 90]:
            features = add_recency_weighted_top_features(features, working, col, top_values, half_life)

    features = add_entropy_features(features, working, EVENT_COL, "event_type")
    for col in CAT_COLS:
        features = add_entropy_features(features, working, col, col)
    if "currency" in working.columns:
        features = add_entropy_features(features, working, "currency", "currency")

    for days in [7, 14, 30, 60, 90]:
        prefix = f"last_{days}d_prefix"
        subset = working[working["days_before_prefix_end"] <= days]
        features = add_subset_aggregates(features, subset, prefix)
        denom_col = f"{prefix}_event_count"
        for col, top_values in fitted_top_values:
            features = add_top_count_features(
                features, subset, col, top_values, prefix=f"{prefix}_{col}", add_counts=False, denom_col=denom_col
            )

    for n_tail in [5, 10, 25, 50]:
        prefix = f"tail_{n_tail}tx_prefix"
        subset = working.groupby(GROUP_COL, sort=False).tail(n_tail)
        features = add_subset_aggregates(features, subset, prefix)
        denom_col = f"{prefix}_event_count"
        for col, top_values in fitted_top_values:
            features = add_top_count_features(
                features, subset, col, top_values, prefix=f"{prefix}_{col}", add_counts=False, denom_col=denom_col
            )

    first_half = working[~working["is_last_half"]]
    last_half = working[working["is_last_half"]]
    drift_base = features.copy()
    drift_base = add_subset_aggregates(drift_base, first_half, "first_half_prefix")
    drift_base = add_subset_aggregates(drift_base, last_half, "last_half_prefix")
    for metric in ["event_count", "amount_sum", "amount_mean", "amount_abs_sum", "positive_share_num", "negative_share_num", "unique_event_types", "active_days"]:
        a = drift_base.get(f"last_half_prefix_{metric}", pd.Series(0.0, index=features.index)).fillna(0.0)
        b = drift_base.get(f"first_half_prefix_{metric}", pd.Series(0.0, index=features.index)).fillna(0.0)
        features[f"drift_prefix_{metric}"] = a - b
    for col, top_values in fitted_top_values:
        first_base = pd.DataFrame({"event_count": first_half.groupby(GROUP_COL).size().reindex(features.index).fillna(0.0)}, index=features.index)
        last_base = pd.DataFrame({"event_count": last_half.groupby(GROUP_COL).size().reindex(features.index).fillna(0.0)}, index=features.index)
        first_tmp = add_top_count_features(first_base, first_half, col, top_values, prefix=f"first_half_prefix_{col}", add_counts=False)
        last_tmp = add_top_count_features(last_base, last_half, col, top_values, prefix=f"last_half_prefix_{col}", add_counts=False)
        for value in top_values:
            first_col = f"first_half_prefix_{col}_top_{safe_name(value)}_freq"
            last_col = f"last_half_prefix_{col}_top_{safe_name(value)}_freq"
            features[f"drift_prefix_{col}_top_{safe_name(value)}_freq"] = last_tmp[last_col].fillna(0.0) - first_tmp[first_col].fillna(0.0)

    top_transitions = train_df["transition"].dropna().value_counts().head(80).index.tolist()
    if top_transitions:
        trans = working.dropna(subset=["transition"])
        trans = trans[trans["transition"].isin(top_transitions)]
        trans_counts = pd.crosstab(trans[GROUP_COL], trans["transition"]).reindex(index=features.index, columns=top_transitions, fill_value=0)
        trans_counts.columns = [f"transition_top_{safe_name(v)}_cnt" for v in top_transitions]
        trans_freqs = trans_counts.div(features["event_count"].replace(0, np.nan), axis=0).fillna(0.0)
        trans_freqs.columns = [c.replace("_cnt", "_freq") for c in trans_counts.columns]
        features = features.join(trans_counts).join(trans_freqs)

    features = features.reindex(all_ids).replace([np.inf, -np.inf], np.nan)
    cat_feature_names = [col for col in cat_feature_names if col in features.columns]
    numeric_cols = [col for col in features.columns if col not in cat_feature_names]
    features[numeric_cols] = features[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)
    for col in cat_feature_names:
        features[col] = features[col].fillna("<NA>").astype(str)
    features.index.name = GROUP_COL

    globals()["CATBOOST_CATEGORICAL_FEATURES"] = cat_feature_names
    globals()["CATBOOST_FITTED_FEATURE_METADATA"] = {
        "top_values": {col: values for col, values in fitted_top_values},
        "top_transitions": top_transitions,
        "reference_time": str(reference_time),
        "raw_categorical_columns": raw_cat_cols,
    }
    return features


features = build_aggregate_features(df_prefix, all_entity_ids, splits["train"])
assert_catboost_feature_frame_is_leak_safe(features, df_prefix, splits, forecast_cut_by_entity)
y = pd.Series(labels_by_entity, name="target").reindex(features.index).astype(int)
print("Features:", features.shape)
print("CatBoost categorical features:", CATBOOST_CATEGORICAL_FEATURES)
features.head()


In [ ]:
# Cell 6 - Low-label CatBoost runs and standardized outputs
iterations = 100 if SMOKE_RUN else 3000
od_wait = 20 if SMOKE_RUN else 200
run_rows = []
prediction_rows = []
feature_importance_rows = []
cat_feature_names = [col for col in CATBOOST_CATEGORICAL_FEATURES if col in features.columns]

X_val = features.loc[splits["val"]]
y_val = y.loc[splits["val"]]
X_test = features.loc[splits["test"]]
y_test = y.loc[splits["test"]]
val_pool = Pool(X_val, y_val, cat_features=cat_feature_names)
test_pool = Pool(X_test, y_test, cat_features=cat_feature_names)

for fraction in LABEL_FRACTIONS:
    for seed in LABEL_SAMPLING_SEEDS:
        train_ids = sample_labeled_entities(splits["train"], labels_by_entity, fraction, seed)
        X_train = features.loc[train_ids]
        y_train = y.loc[train_ids]
        train_pool = Pool(X_train, y_train, cat_features=cat_feature_names)

        print(f"fraction={fraction:.2f} seed={seed} train_entities={len(train_ids)}")
        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=iterations,
            learning_rate=0.025,
            depth=6,
            l2_leaf_reg=6.0,
            random_seed=seed,
            auto_class_weights="Balanced",
            bootstrap_type="Bayesian",
            bagging_temperature=0.5,
            random_strength=1.0,
            od_type="Iter",
            od_wait=od_wait,
            allow_writing_files=False,
            verbose=100,
        )
        model.fit(train_pool, eval_set=val_pool, use_best_model=True)

        feature_importance_rows.append(pd.DataFrame({
            "feature": X_train.columns,
            "importance": model.get_feature_importance(train_pool),
            "pipeline": "aggregates_catboost",
            "fraction": fraction,
            "seed": seed,
        }))

        test_proba = model.predict_proba(test_pool)[:, 1]
        metrics = compute_binary_metrics(y_test.values, test_proba)
        row = {
            "baseline": "aggregates_catboost",
            "pipeline": "aggregates_catboost",
            "fraction": fraction,
            "seed": seed,
            "train_entities": len(train_ids),
            "best_iteration": int(model.get_best_iteration() or 0),
            **metrics,
        }
        run_rows.append(row)
        print(row)

        run_pred = pd.DataFrame({
            GROUP_COL: X_test.index,
            "target": y_test.values,
            "proba": test_proba,
            "baseline": "aggregates_catboost",
            "pipeline": "aggregates_catboost",
            "fraction": fraction,
            "seed": seed,
        })
        prediction_rows.append(run_pred)

feature_importance_df = pd.concat(feature_importance_rows, ignore_index=True)
fi_path = OUTPUT_DIR / "feature_importance.csv"
feature_importance_df.to_csv(fi_path, index=False)
print(f"Saved: {fi_path}")

fig, ax = plt.subplots(figsize=(8, 7))
fi_top = feature_importance_df.groupby("feature")["importance"].mean().sort_values().tail(30)
ax.barh(fi_top.index, fi_top.values)
ax.set_title("CatBoost mean feature importance")
ax.set_xlabel("Importance")
fi_plot_path = OUTPUT_DIR / "plots" / "feature_importance_top30.png"
fi_plot_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fi_plot_path, dpi=160, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {fi_plot_path}")

all_runs_df, low_label_agg_df, summary_df, predictions_df, metrics_payload = save_low_label_outputs(
    run_rows,
    prediction_rows,
    baseline_name="aggregates_catboost",
    output_dir=OUTPUT_DIR,
    extra_artifacts={
        "feature_importance": fi_path,
        "feature_importance_plot": fi_plot_path,
        "cat_features": cat_feature_names,
        "feature_metadata": CATBOOST_FITTED_FEATURE_METADATA,
    },
)

try:
    display(low_label_agg_df)
except NameError:
    print(low_label_agg_df.to_string(index=False))
